In [17]:
import numpy as np
import pandas as pd
import pywt
from scipy.stats import entropy
from scipy.signal import find_peaks
from sklearn.preprocessing import StandardScaler

# 提取三个特征模块

输出：

features_module1_dwt：DWT 多尺度特征

features_module2_phase：24h 相位特征

features_module3_amplitude：幅值/消耗特征

features_fused：加权融合后的最终特征空间

## 0.读取数据

In [18]:
# ----------------------------------------------------
# 1. 读取采样后的 CSV（每一列是一个负荷节点）
# ----------------------------------------------------
CSV_PATH = "../k-Shape/kshape_results_1440_0Start/sampled_1440_0Start.csv"
PERIOD = 24

df = pd.read_csv(CSV_PATH, header=0)

# ----------------------------------------------------
# 2. 周期对齐缺失值填补（24h）
# ----------------------------------------------------
missing = df.isna().sum().sum()

if missing > 0:
    print(f"发现 {missing} 个缺失值，使用周期对齐插值 (period={PERIOD}) 填补。")

    for col_idx, col_name in enumerate(df.columns):
        series = df[col_name].values
        nan_idx = np.where(np.isnan(series))[0]

        if len(nan_idx) == 0:
            continue

        print(f"列 `{col_name}` (第 {col_idx + 1} 列) 缺失位置: {nan_idx.tolist()}")

        # -------- 单点缺失 --------
        if len(nan_idx) == 1:
            t = nan_idx[0]
            series[t] = 0.5 * (series[t - PERIOD] + series[t + PERIOD])

        # -------- 两个连续缺失 --------
        elif len(nan_idx) == 2 and nan_idx[1] == nan_idx[0] + 1:
            t0, t1 = nan_idx
            series[t0] = 0.5 * (series[t0 - PERIOD] + series[t0 + PERIOD])
            series[t1] = 0.5 * (series[t1 - PERIOD] + series[t1 + PERIOD])

        else:
            raise ValueError(
                f"列 {col_name} 出现不符合假设的缺失模式: {nan_idx}"
            )

        df[col_name] = series
else:
    print("未发现缺失值，无需填补。")

# ----------------------------------------------------
# 3. 构造与你原来完全一致的变量
# ----------------------------------------------------
data = df.values.T        # shape: (n_nodes, 1440)
node_names = df.columns.tolist()
n_nodes = data.shape[0]

print("数据读取与缺失值填补完成：")
print("data shape:", data.shape)
print("n_nodes:", n_nodes)

发现 3 个缺失值，使用周期对齐插值 (period=24) 填补。
列 `Robin_office_Soledad` (第 42 列) 缺失位置: [825]
列 `Mouse_health_Modesto` (第 63 列) 缺失位置: [845, 846]
数据读取与缺失值填补完成：
data shape: (68, 1440)
n_nodes: 68


## 模块 1：DWT 多尺度特征（周期 & 波动形态）

提取内容（对DWT分解后的每一层提取）：

mean

std

energy

entropy

输出维度：维度 = (7 层 + 1 个 A_L) × 4 = 32 维

In [20]:
def extract_dwt_features(
    ts,
    wavelet="sym5",
    level=7
):
    """
    对单条时间序列提取 DWT 多尺度统计特征
    """
    coeffs = pywt.wavedec(ts, wavelet=wavelet, level=level)

    features = []

    # coeffs: [A_L, D_L, D_{L-1}, ..., D_1]
    for c in coeffs:
        c = np.asarray(c)

        mean_c = np.mean(c)
        std_c = np.std(c)
        energy_c = np.sum(c ** 2)

        # 归一化后算熵（防止数值问题）
        prob = np.abs(c)
        prob = prob / (np.sum(prob) + 1e-12)
        entropy_c = entropy(prob)

        features.extend([mean_c, std_c, energy_c, entropy_c])

    return np.array(features)


In [21]:
features_module1_dwt = np.vstack([
    extract_dwt_features(data[i])
    for i in range(n_nodes)
])

print("Module 1 (DWT) feature shape:", features_module1_dwt.shape)


Module 1 (DWT) feature shape: (68, 32)


## 模块 2：24h 日内相位特征

核心思想：

1440 h → reshape 为 (60 天 × 24 h)

得到“典型日负荷曲线”

在 24h 曲线上提取峰谷与相位信息

输出维度：6
（peak_hour,
valley_hour,
peak_value,
valley_value,
num_peaks,
peak_to_valley_ratio）

In [22]:
def extract_phase_features(ts):
    """
    提取日内相位与峰谷特征
    """
    ts = ts.reshape(-1, 24)   # (60, 24)
    daily_profile = ts.mean(axis=0)  # 典型 24h 曲线

    # 主峰检测
    peaks, _ = find_peaks(daily_profile)
    valleys, _ = find_peaks(-daily_profile)

    # 主峰
    if len(peaks) > 0:
        main_peak_idx = peaks[np.argmax(daily_profile[peaks])]
        peak_hour = main_peak_idx
        peak_value = daily_profile[main_peak_idx]
        num_peaks = len(peaks)
    else:
        peak_hour = -1
        peak_value = daily_profile.max()
        num_peaks = 0

    # 主谷
    if len(valleys) > 0:
        main_valley_idx = valleys[np.argmin(daily_profile[valleys])]
        valley_hour = main_valley_idx
        valley_value = daily_profile[main_valley_idx]
    else:
        valley_hour = -1
        valley_value = daily_profile.min()

    # 峰谷比
    peak_to_valley_ratio = (
        peak_value / (valley_value + 1e-6)
        if valley_value != 0 else 0
    )

    return np.array([
        peak_hour,
        valley_hour,
        peak_value,
        valley_value,
        num_peaks,
        peak_to_valley_ratio
    ])


In [23]:
features_module2_phase = np.vstack([
    extract_phase_features(data[i])
    for i in range(n_nodes)
])

print("Module 2 (Phase) feature shape:", features_module2_phase.shape)


Module 2 (Phase) feature shape: (68, 6)


## 模块 3：幅值 & 消耗水平特征

提取内容：

mean / max / min

p95 / p5

range

coefficient of variation

输出维度：7

In [24]:
def extract_amplitude_features(ts):
    mean_load = np.mean(ts)
    std_load = np.std(ts)
    max_load = np.max(ts)
    min_load = np.min(ts)

    p95 = np.percentile(ts, 95)
    p5 = np.percentile(ts, 5)

    load_range = max_load - min_load
    cv = std_load / (mean_load + 1e-6)

    return np.array([
        mean_load,
        max_load,
        min_load,
        p95,
        p5,
        load_range,
        cv
    ])


In [25]:
features_module3_amplitude = np.vstack([
    extract_amplitude_features(data[i])
    for i in range(n_nodes)
])

print("Module 3 (Amplitude) feature shape:", features_module3_amplitude.shape)


Module 3 (Amplitude) feature shape: (68, 7)


## 模块 4：模块内标准化 + 加权融合

融合出用于聚类的特征空间

标准化：

In [27]:
scaler_dwt = StandardScaler()
scaler_phase = StandardScaler()
scaler_amp = StandardScaler()

features_dwt_std = scaler_dwt.fit_transform(features_module1_dwt)
features_phase_std = scaler_phase.fit_transform(features_module2_phase)
features_amp_std = scaler_amp.fit_transform(features_module3_amplitude)


权重设置

In [28]:
w_dwt = 1.0
w_phase = 1.0
w_amp = 1.0


融合为45维特征空间

In [29]:
features_fused = np.hstack([
    w_dwt * features_dwt_std,
    w_phase * features_phase_std,
    w_amp * features_amp_std
])

print("Final fused feature shape:", features_fused.shape)


Final fused feature shape: (68, 45)


输出为DF，保存为csv

In [31]:
features_fused_df = pd.DataFrame(
    features_fused,
    index=node_names
)

features_fused_df.to_csv("Multi-Feature_result/fused_features_for_clustering.csv")
